# Colon w/ TANDEM+CatBoost+LogReg+TabPFN-2.5

In [ ]:
# ============================================================
# HDLSS 5x5 CV (TSTR / TRTR / C2ST)
# with TabPFN-2.5, TANDEM, CatBoost, Logistic Regression
# using a *fold-wise* BSTabDiff generator (train-only per fold)
# + Fidelity diagnostics (real vs synthetic, global generator)
# + BSTabDiff runtime + GPU/CPU usage
# ============================================================

import os
import math
import time
import random
import inspect
import numpy as np
import pandas as pd

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_auc_score, accuracy_score
from sklearn.utils import check_random_state
from sklearn.feature_selection import mutual_info_classif
from sklearn.neighbors import NearestNeighbors

# Optional CPU memory stats
try:
    import psutil
except Exception:
    psutil = None

# ----------------------------
# Optional: TabPFN imports
# ----------------------------
try:
    from tabpfn import TabPFNClassifier
except Exception:
    TabPFNClassifier = None

# ----------------------------
# Optional: CatBoost imports
# ----------------------------
try:
    from catboost import CatBoostClassifier
except Exception:
    CatBoostClassifier = None

# ----------------------------
# Optional: torch + TANDEM imports
# ----------------------------
try:
    import torch
    import torch.nn as nn
    import torch.nn.functional as F
    from torch.utils.data import DataLoader, TensorDataset
except Exception:
    torch = None
    nn = None
    F = None
    DataLoader = None
    TensorDataset = None

# Default device for torch-based models (TANDEM, generator)
if torch is not None and torch.cuda.is_available():
    DEFAULT_TORCH_DEVICE = torch.device("cuda:0")
else:
    DEFAULT_TORCH_DEVICE = None

if torch is not None:
    try:
        torch.set_float32_matmul_precision("medium")
    except Exception:
        pass

GLOBAL_SEED = 42


def set_seed(seed=GLOBAL_SEED):
    random.seed(seed)
    np.random.seed(seed)
    if torch is not None and torch.cuda.is_available():
        torch.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)


# ============================================================
# 1) TabPFN builder (robust to version differences)
# ============================================================
def _make_tabpfn(seed=0, tabpfn_device=None, tabpfn_ensembles=16):
    """
    Handles both APIs:
      - PriorLabs TabPFN: TabPFNClassifier(n_estimators=..., device=...)
      - Older TabPFN:     TabPFNClassifier(N_ensemble_configurations=..., device=...)
    """
    if TabPFNClassifier is None:
        raise ImportError("TabPFN not installed. Try: pip install tabpfn")

    # Auto device if not provided
    if tabpfn_device is None:
        if (torch is not None) and torch.cuda.is_available():
            tabpfn_device = "cuda:0"
        else:
            tabpfn_device = "cpu"

    sig = inspect.signature(TabPFNClassifier)
    params = sig.parameters

    kwargs = {}

    # Ensemble count (API differs by version)
    if "n_estimators" in params:
        kwargs["n_estimators"] = int(tabpfn_ensembles)
    elif "N_ensemble_configurations" in params:
        kwargs["N_ensemble_configurations"] = int(tabpfn_ensembles)

    # Device (often "auto"/"cpu"/"cuda"/"cuda:0")
    if "device" in params:
        kwargs["device"] = str(tabpfn_device)

    # Seed / random_state
    if "seed" in params:
        kwargs["seed"] = int(seed)
    elif "random_state" in params:
        kwargs["random_state"] = int(seed)

    return TabPFNClassifier(**kwargs)


# ============================================================
# 2) TANDEM components (BN-free enc/dec for tiny HDLSS batches)
# ============================================================

if torch is not None:
    class GatingNet(nn.Module):
        def __init__(self, input_dim, hidden_dim=128, sigma=0.5):
            super().__init__()
            self.input_dim = input_dim
            self.hidden_dim = hidden_dim
            self.sigma = sigma
            self.net = nn.Sequential(
                nn.Linear(input_dim, hidden_dim),
                nn.Tanh(),
                nn.Linear(hidden_dim, input_dim),
            )
            self.apply(self.init_weights)
            self._sqrt_2 = math.sqrt(2)

        @staticmethod
        def init_weights(m):
            if isinstance(m, nn.Linear):
                nn.init.normal_(m.weight, std=0.001)
                if hasattr(m, "bias") and m.bias is not None:
                    m.bias.data.fill_(0.1)

        def forward(self, x):
            noise = torch.normal(0, self.sigma, size=x.size(), device=x.device) if self.training else 0
            mu = self.net(x)
            z = mu + 0.1 * noise * self.training
            gates = self.hard_sigmoid(z)
            sparse_x = x * gates
            return mu, sparse_x, gates

        @staticmethod
        def hard_sigmoid(x):
            return torch.clamp(x + 0.5, 0.0, 1.0)

        def regularization(self, mu, reduction_func=torch.sum):
            return reduction_func(0.5 + 0.5 * torch.erf(mu / (self.sigma * self._sqrt_2)))

        def get_gates(self, x):
            with torch.no_grad():
                gates = self.hard_sigmoid(self.net(x))
            return gates

        def num_open_gates(self, x):
            return self.get_gates(x).sum(dim=1).cpu().median(dim=0)[0].item()


    class OSDT(nn.Module):
        def __init__(self, input_dim, depth, gating_net=None):
            super().__init__()
            self.input_dim = input_dim
            self.depth = depth
            self.gating_net = gating_net
            self.split_weights = nn.Parameter(torch.randn(depth, input_dim))
            self.feature_thresholds = nn.Parameter(torch.randn(depth))
            self.log_temperatures = nn.Parameter(torch.zeros(depth))
            indices = torch.arange(2**depth)
            bits = ((indices[:, None] & (1 << torch.arange(depth))) > 0).float()
            self.register_buffer("bin_codes", bits.t())

        def forward(self, x):
            all_selected = []
            for i in range(self.depth):
                x_gated = self.gating_net(x)[1] if self.gating_net is not None else x
                split_score = torch.einsum("bi,i->b", x_gated, self.split_weights[i])
                all_selected.append(split_score)
            selected_features = torch.stack(all_selected, dim=1)
            logits = (selected_features - self.feature_thresholds) * torch.exp(-self.log_temperatures)
            logits = torch.stack([-logits, logits], dim=-1)
            bins = torch.sigmoid(logits)
            bins_left = bins[..., 0]
            bins_right = bins[..., 1]
            bin_codes = self.bin_codes.unsqueeze(0)
            bin_match = bins_left.unsqueeze(-1) * (1 - bin_codes) + bins_right.unsqueeze(-1) * bin_codes
            path_probs = bin_match.prod(dim=-2)
            return path_probs


    class OSDTEncoder(nn.Module):
        def __init__(self, input_dim, num_trees, depth, gating_net=None):
            super().__init__()
            self.trees = nn.ModuleList(
                [OSDT(input_dim, depth, gating_net=gating_net) for _ in range(num_trees)]
            )
            self.output_dim = 2**depth

        def forward(self, x):
            outputs = [tree(x) for tree in self.trees]
            outputs = torch.stack(outputs, dim=0)
            return outputs.mean(dim=0)


    class ModularEncoder(nn.Module):
        """
        Simple MLP encoder WITHOUT BatchNorm to avoid batch_size=1 issues
        on very small HDLSS splits.
        """
        def __init__(self, input_dim, hidden_layers):
            super().__init__()
            layers = []
            current_in = input_dim
            for h in hidden_layers:
                layers.append(nn.Linear(current_in, h))
                layers.append(nn.LeakyReLU())
                current_in = h
            self.encoder = nn.Sequential(*layers)
            self.output_dim = hidden_layers[-1]

        def forward(self, x):
            return self.encoder(x)


    class ModularDecoder(nn.Module):
        """
        Decoder mirrors encoder, also WITHOUT BatchNorm.
        """
        def __init__(self, output_dim, hidden_layers):
            super().__init__()
            layers = []
            current_in = hidden_layers[-1]
            for h in reversed(hidden_layers[:-1]):
                layers.append(nn.Linear(current_in, h))
                layers.append(nn.LeakyReLU())
                current_in = h
            layers.append(nn.Linear(current_in, output_dim))
            self.decoder = nn.Sequential(*layers)

        def forward(self, z):
            return self.decoder(z)


    class Tandem(nn.Module):
        """
        TANDEM: Both encoders must end in 2**osdt_depth; decoder expects same latent size.
        Returns both reconstructions and both latent vectors.
        """
        def __init__(self, input_dim, hidden_layers, num_trees=1, osdt_depth=7, gating_net=None):
            super().__init__()
            assert hidden_layers[-1] == 2**osdt_depth, (
                f"The final latent dimension ({hidden_layers[-1]}) must equal 2**osdt_depth ({2**osdt_depth})."
            )
            self.nn_encoder = ModularEncoder(input_dim, hidden_layers)
            self.osdt_encoder = OSDTEncoder(input_dim, num_trees, osdt_depth, gating_net=gating_net)
            self.decoder = ModularDecoder(input_dim, hidden_layers)

        def forward(self, x):
            z_nn = self.nn_encoder(x)
            z_osdt = self.osdt_encoder(x)
            x_rec_nn = self.decoder(z_nn)
            x_rec_osdt = self.decoder(z_osdt)
            return x_rec_nn, x_rec_osdt, z_nn, z_osdt


    class TandemModel(nn.Module):
        def __init__(
            self,
            input_dim: int,
            n_classes: int,
            hidden_layers=None,
            num_trees: int = 1,
            osdt_depth: int = 7,
            use_gating: bool = True,
            gating_hidden_dim: int = 128,
            gating_sigma: float = 0.5,
        ):
            super().__init__()
            if hidden_layers is None:
                last_dim = 2**osdt_depth
                hidden_layers = [last_dim * 2, last_dim]
            assert hidden_layers[-1] == 2**osdt_depth, (
                f"hidden_layers[-1] must equal 2**osdt_depth ({2**osdt_depth})"
            )

            gating_net = (
                GatingNet(input_dim, hidden_dim=gating_hidden_dim, sigma=gating_sigma)
                if use_gating
                else None
            )
            self.tandem = Tandem(
                input_dim=input_dim,
                hidden_layers=hidden_layers,
                num_trees=num_trees,
                osdt_depth=osdt_depth,
                gating_net=gating_net,
            )
            self.classifier = nn.Linear(hidden_layers[-1], n_classes)

        def forward(self, x):
            """
            Returns:
              logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt
            """
            x_rec_nn, x_rec_osdt, z_nn, z_osdt = self.tandem(x)
            logits = self.classifier(z_nn)
            return logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt


    class TandemClassifier:
        """
        Sklearn-style wrapper around TandemModel.
        Uses an internal 80/20 stratified train/val split if X_val / y_val are not provided.
        """

        def __init__(
            self,
            hidden_layers=None,
            num_trees: int = 1,
            osdt_depth: int = 7,
            use_gating: bool = True,
            gating_hidden_dim: int = 128,
            gating_sigma: float = 0.5,
            lr: float = 1e-3,
            weight_decay: float = 1e-5,
            batch_size: int = 32,
            max_epochs: int = 100,
            patience: int = 10,
            rec_weight: float = 1.0,
            align_weight: float = 0.1,
            random_state: int = 42,
            device=None,
        ):
            self.hidden_layers = hidden_layers
            self.num_trees = num_trees
            self.osdt_depth = osdt_depth
            self.use_gating = use_gating
            self.gating_hidden_dim = gating_hidden_dim
            self.gating_sigma = gating_sigma

            self.lr = lr
            self.weight_decay = weight_decay
            self.batch_size = batch_size
            self.max_epochs = max_epochs
            self.patience = patience
            self.rec_weight = rec_weight
            self.align_weight = align_weight
            self.random_state = random_state

            if device is None:
                if DEFAULT_TORCH_DEVICE is not None:
                    self.device = DEFAULT_TORCH_DEVICE
                else:
                    self.device = torch.device("cpu")
            else:
                self.device = device

            self.model = None
            self.classes_ = None

        def _build_model(self, n_features: int, n_classes: int):
            self.model = TandemModel(
                input_dim=n_features,
                n_classes=n_classes,
                hidden_layers=self.hidden_layers,
                num_trees=self.num_trees,
                osdt_depth=self.osdt_depth,
                use_gating=self.use_gating,
                gating_hidden_dim=self.gating_hidden_dim,
                gating_sigma=self.gating_sigma,
            ).to(self.device)

        def fit(self, X, y, X_val=None, y_val=None):
            """
            If X_val / y_val are None, uses an internal 80/20 stratified split
            for early stopping. That makes it compatible with generic code like
            eval_5x5_tstr_foldwise_gen that only calls clf.fit(X, y).
            """
            set_seed(self.random_state)

            X = np.asarray(X, dtype=np.float32)
            y = np.asarray(y)

            if X_val is None or y_val is None:
                from sklearn.model_selection import StratifiedShuffleSplit

                splitter = StratifiedShuffleSplit(
                    n_splits=1, test_size=0.2, random_state=self.random_state
                )
                (tr_idx, va_idx), = splitter.split(X, y)
                X_tr, X_val = X[tr_idx], X[va_idx]
                y_tr, y_val = y[tr_idx], y[va_idx]
            else:
                X_tr, X_val = np.asarray(X, dtype=np.float32), np.asarray(X_val, dtype=np.float32)
                y_tr, y_val = np.asarray(y), np.asarray(y_val)

            n_samples, n_features = X_tr.shape
            classes = np.unique(y_tr)
            self.classes_ = classes
            n_classes = len(classes)

            class_to_idx = {c: i for i, c in enumerate(classes)}
            y_tr_idx = np.vectorize(class_to_idx.get)(y_tr).astype(np.int64)
            y_val_idx = np.vectorize(class_to_idx.get)(y_val).astype(np.int64)

            self._build_model(n_features, n_classes)
            assert self.model is not None
            model = self.model

            optimizer = torch.optim.Adam(model.parameters(), lr=self.lr, weight_decay=self.weight_decay)
            ce_loss_fn = nn.CrossEntropyLoss()

            X_tr_t = torch.from_numpy(X_tr)
            y_tr_t = torch.from_numpy(y_tr_idx)
            X_val_t = torch.from_numpy(X_val)
            y_val_t = torch.from_numpy(y_val_idx)

            train_ds = TensorDataset(X_tr_t, y_tr_t)
            val_ds = TensorDataset(X_val_t, y_val_t)

            train_loader = DataLoader(
                train_ds,
                batch_size=self.batch_size,
                shuffle=True,
                drop_last=False,
            )
            val_loader = DataLoader(
                val_ds,
                batch_size=self.batch_size,
                shuffle=False,
                drop_last=False,
            )

            best_state = None
            best_val_loss = float("inf")
            epochs_no_improve = 0

            model.to(self.device)

            for epoch in range(1, self.max_epochs + 1):
                # ---- Train ----
                model.train()
                train_loss_sum = 0.0
                n_train = 0

                for xb, yb in train_loader:
                    xb = xb.to(self.device)
                    yb = yb.to(self.device)

                    optimizer.zero_grad()
                    logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt = model(xb)

                    ce = ce_loss_fn(logits, yb)
                    rec_nn = F.mse_loss(x_rec_nn, xb)
                    rec_osdt = F.mse_loss(x_rec_osdt, xb)
                    align = F.mse_loss(z_nn, z_osdt)

                    loss = ce + self.rec_weight * (rec_nn + rec_osdt) + self.align_weight * align
                    loss.backward()
                    nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                    optimizer.step()

                    bs = xb.size(0)
                    train_loss_sum += loss.item() * bs
                    n_train += bs

                train_loss = train_loss_sum / max(n_train, 1)

                # ---- Validation ----
                model.eval()
                val_loss_sum = 0.0
                n_val = 0
                with torch.no_grad():
                    for xb, yb in val_loader:
                        xb = xb.to(self.device)
                        yb = yb.to(self.device)
                        logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt = model(xb)

                        ce = ce_loss_fn(logits, yb)
                        rec_nn = F.mse_loss(x_rec_nn, xb)
                        rec_osdt = F.mse_loss(x_rec_osdt, xb)
                        align = F.mse_loss(z_nn, z_osdt)

                        loss = ce + self.rec_weight * (rec_nn + rec_osdt) + self.align_weight * align

                        bs = xb.size(0)
                        val_loss_sum += loss.item() * bs
                        n_val += bs

                val_loss = val_loss_sum / max(n_val, 1)

                if val_loss < best_val_loss - 1e-6:
                    best_val_loss = val_loss
                    best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                    epochs_no_improve = 0
                else:
                    epochs_no_improve += 1
                    if epochs_no_improve >= self.patience:
                        break

            if best_state is not None:
                model.load_state_dict(best_state)
                model.to(self.device)

            return self

        def _predict_logits(self, X):
            if self.model is None:
                raise RuntimeError("Model not fitted yet.")

            X = np.asarray(X, dtype=np.float32)
            X_t = torch.from_numpy(X)
            ds = TensorDataset(X_t)
            loader = DataLoader(ds, batch_size=self.batch_size, shuffle=False)

            self.model.eval()
            self.model.to(self.device)

            all_logits = []
            with torch.no_grad():
                for (xb,) in loader:
                    xb = xb.to(self.device)
                    logits, x_rec_nn, x_rec_osdt, z_nn, z_osdt = self.model(xb)
                    all_logits.append(logits.cpu())
            logits = torch.cat(all_logits, dim=0).numpy()
            return logits

        def predict_proba(self, X):
            logits = self._predict_logits(X)
            probs = F.softmax(torch.from_numpy(logits), dim=1).numpy()
            return probs

        def predict(self, X):
            probs = self.predict_proba(X)
            idx = probs.argmax(axis=1)
            if self.classes_ is None:
                return idx
            return self.classes_[idx]


# ============================================================
# 3) Classifiers (4 options)
# ============================================================
def make_clf(
    name="lr",
    seed=0,
    # TabPFN knobs
    tabpfn_device=None,  # None -> auto, or "cpu"/"cuda"/"cuda:0"
    tabpfn_ensembles=16,
    tabpfn_kbest=None,  # for 2000+ features (Colon)
):
    """
    Returns an estimator with predict_proba (or calibrated probas).

    Allowed names:
      - "lr"       : Logistic Regression
      - "tabpfn"   : TabPFN-2.5 (via TabPFNClassifier)
      - "catboost" : CatBoost (GPU on device 0 if available)
      - "tandem"   : TANDEM (PyTorch, cuda:0)
    """
    name = name.lower().strip()

    # ----- Logistic Regression (CPU, no tuning) -----
    if name == "lr":
        return Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler(with_mean=True, with_std=True)),
                (
                    "clf",
                    LogisticRegression(
                        max_iter=10000,
                        class_weight="balanced",
                        solver="lbfgs",
                        n_jobs=None,
                    ),
                ),
            ]
        )

    # ----- TabPFN-2.5 -----
    if name == "tabpfn":
        from sklearn.feature_selection import SelectKBest, f_classif

        steps = [("imp", SimpleImputer(strategy="median"))]

        # Optional feature reduction for Colon (2000 feats) etc.
        if tabpfn_kbest is not None:
            steps.append(("kbest", SelectKBest(score_func=f_classif, k=int(tabpfn_kbest))))

        steps.append(
            (
                "clf",
                _make_tabpfn(
                    seed=seed,
                    tabpfn_device=tabpfn_device,
                    tabpfn_ensembles=tabpfn_ensembles,
                ),
            )
        )
        return Pipeline(steps)

    # ----- CatBoost (GPU if available) -----
    if name == "catboost":
        if CatBoostClassifier is None:
            raise ImportError("CatBoost not installed. Try: pip install catboost")

        use_gpu = (torch is not None) and torch.cuda.is_available()
        params = dict(
            depth=6,
            learning_rate=0.05,
            loss_function="Logloss",
            eval_metric="Logloss",  # avoid GPU AUC warnings
            iterations=1000,
            random_seed=seed,
            verbose=False,  # only 'verbose' set; no logging_level/silent clash
        )
        if use_gpu:
            params.update(task_type="GPU", devices="0")

        cb = CatBoostClassifier(**params)

        return Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                # CatBoost doesn't need scaling for numeric features
                ("clf", cb),
            ]
        )

    # ----- TANDEM (PyTorch) -----
    if name == "tandem":
        if torch is None or TandemClassifier is None:
            raise ImportError("PyTorch / TANDEM not available.")

        if DEFAULT_TORCH_DEVICE is not None:
            dev = DEFAULT_TORCH_DEVICE
        else:
            dev = torch.device("cpu")

        clf = TandemClassifier(
            hidden_layers=None,
            num_trees=1,
            osdt_depth=7,  # latent dim = 128
            use_gating=True,
            gating_hidden_dim=128,
            gating_sigma=0.5,
            lr=1e-3,
            weight_decay=1e-5,
            batch_size=32,
            max_epochs=100,
            patience=10,
            rec_weight=1.0,
            align_weight=0.1,
            random_state=seed,
            device=dev,
        )

        return Pipeline(
            [
                ("imp", SimpleImputer(strategy="median")),
                ("sc", StandardScaler()),
                ("clf", clf),
            ]
        )

    raise ValueError(f"Unknown classifier '{name}'. Choose from: lr, tabpfn, catboost, tandem")


# ============================================================
# 4) Utility helpers
# ============================================================
def _predict_proba_binary(clf, X):
    """
    Returns P(y=1) for binary problems.
    Assumes classifier has predict_proba or decision_function.
    """
    if hasattr(clf, "predict_proba"):
        proba = clf.predict_proba(X)
        # Handle shape (N,2) or (N,) gracefully
        if proba.ndim == 2 and proba.shape[1] >= 2:
            return proba[:, 1]
        return proba
    if hasattr(clf, "decision_function"):
        s = clf.decision_function(X)
        return 1.0 / (1.0 + np.exp(-s))
    raise ValueError("Classifier has neither predict_proba nor decision_function.")


# ============================================================
# 5) Strict 5x5 CV with fold-wise generator training
# ============================================================
def eval_5x5_tstr_foldwise_gen(
    X_real,
    y_real,
    clf_name="lr",  # "lr", "tabpfn", "catboost", "tandem"
    generator_fit_fn=None,  # callable returned by make_generator_fit_fn
    n_syn=200,  # synthetic samples per fold
    do_trtr=True,
    do_c2st=True,
    seed=0,
    # TabPFN knobs (used only if clf_name == "tabpfn")
    tabpfn_device=None,
    tabpfn_ensembles=16,
    tabpfn_kbest=200,  # for Colon (2000 feats); set None to disable
):
    """
    Strict 5x5 CV where, for each fold:
      - The generator is fit ONLY on that fold's training set (Xtr, ytr).
      - Synthetic data are sampled from this fold-specific generator.
      - TSTR: train classifier on synthetic, test on held-out real Xte.
      - TRTR: train/test on real (standard).
      - C2ST: discriminate that fold's real training samples vs that fold's synthetic samples.

    This avoids any leakage of test folds into the generator.
    """
    if generator_fit_fn is None:
        raise ValueError("generator_fit_fn must be provided (e.g., from make_generator_fit_fn).")

    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=5, n_repeats=5, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        # ---------- (1) Fit generator on TRAIN ONLY ----------
        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn_fold, R_syn_fold, y_syn_fold = gen_fold.sample(n=n_syn)

        # ---------- (2) TSTR: train on synthetic, test on real ----------
        clf_tstr = make_clf(
            clf_name,
            seed=fold_seed,
            tabpfn_device=tabpfn_device,
            tabpfn_ensembles=tabpfn_ensembles,
            tabpfn_kbest=tabpfn_kbest,
        )
        clf_tstr.fit(X_syn_fold, y_syn_fold)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(roc_auc_score(yte, proba_t))
        tstr_acc.append(accuracy_score(yte, pred_t))

        # ---------- (3) TRTR: standard real train/test ----------
        if do_trtr:
            clf_trtr = make_clf(
                clf_name,
                seed=fold_seed,
                tabpfn_device=tabpfn_device,
                tabpfn_ensembles=tabpfn_ensembles,
                tabpfn_kbest=tabpfn_kbest,
            )
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(roc_auc_score(yte, proba_r))
            trtr_acc.append(accuracy_score(yte, pred_r))

        # ---------- (4) C2ST: real vs synthetic *for this fold* ----------
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn_fold))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn_fold), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn_fold[idx_s]])
            ymix = np.concatenate(
                [np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)]
            )  # 1=real, 0=synth
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf("lr", seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(roc_auc_score(ymix_te, proba_m))
            c2st_acc.append(accuracy_score(ymix_te, pred_m))

        # Optional: per-fold progress print (comment out if noisy)
        print(
            f"[Fold {fold_id:02d}] "
            f"TSTR ACC={tstr_acc[-1]:.4f}, AUC={tstr_auc[-1]:.4f}"
        )

    out = {
        "tstr_auc_mean": float(np.mean(tstr_auc)),
        "tstr_auc_std": float(np.std(tstr_auc)),
        "tstr_acc_mean": float(np.mean(tstr_acc)),
        "tstr_acc_std": float(np.std(tstr_acc)),
        "n_folds": len(tstr_auc),
    }
    if do_trtr and len(trtr_auc) > 0:
        out.update(
            {
                "trtr_auc_mean": float(np.mean(trtr_auc)),
                "trtr_auc_std": float(np.std(trtr_auc)),
                "trtr_acc_mean": float(np.mean(trtr_acc)),
                "trtr_acc_std": float(np.std(trtr_acc)),
            }
        )
    if do_c2st and len(c2st_auc) > 0:
        out.update(
            {
                "c2st_auc_mean": float(np.mean(c2st_auc)),
                "c2st_auc_std": float(np.std(c2st_auc)),
                "c2st_acc_mean": float(np.mean(c2st_acc)),
                "c2st_acc_std": float(np.std(c2st_acc)),
            }
        )

    return out


# ============================================================
# 6) CSV saving helpers
# ============================================================
def save_synth_csv(
    X_syn: np.ndarray,
    y_syn: np.ndarray | None,
    out_csv: str = "colon_synth_BSTabDiff_n200.csv",
    feature_names: list[str] | None = None,
):
    X_syn = np.asarray(X_syn)
    n, m = X_syn.shape

    if feature_names is None:
        feature_names = [f"f{j}" for j in range(m)]
    assert len(feature_names) == m

    df = pd.DataFrame(X_syn, columns=feature_names)
    if y_syn is not None:
        y_syn = np.asarray(y_syn).astype(int)
        df.insert(0, "y", y_syn)

    df.to_csv(out_csv, index=False)
    print(
        f"Saved CSV: {out_csv} | shape={df.shape} | nan_frac={np.isnan(X_syn).mean():.6g}"
    )
    return out_csv


def save_synth_mask_csv(R_syn: np.ndarray, out_csv: str = "colon_synth_BSTabDiff_n200_mask.csv"):
    R_syn = np.asarray(R_syn).astype(int)
    dfR = pd.DataFrame(R_syn, columns=[f"r{j}" for j in range(R_syn.shape[1])])
    dfR.to_csv(out_csv, index=False)
    print(
        f"Saved mask CSV: {out_csv} | shape={dfR.shape} | observed_frac={R_syn.mean():.6g}"
    )
    return out_csv


# ============================================================
# 7) BSTabDiff generator: FIXED config (no Optuna)
# ============================================================
from block_subunit_gen import FeatureSpec, fit_block_subunit_generator


def make_generator_fit_fn(
    M=32,
    prior_type="diffusion",
    device="cuda:0",
    prior_epochs=10000,
    prior_batch=128,
    prior_lr=1e-3,
    ema_decay=0.999,
    verbose_every=0,
):
    """
    Fixed BSTabDiff generator fit fn.
    - prior_epochs=10000 (long training)
    - save_best=True: internally keeps the best checkpoint by its own loss.
    """
    def _fit(Xtr, ytr, seed=0):
        feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(Xtr.shape[1])]
        gen = fit_block_subunit_generator(
            X=Xtr,
            feature_specs=feature_specs,
            y=ytr,
            M=M,
            blocks=None,
            permute_features=False,
            prior_type=prior_type,
            device=device,
            seed=seed,
            prior_epochs=prior_epochs,
            prior_batch=prior_batch,
            prior_lr=prior_lr,
            verbose_every=verbose_every,
            save_dir=None,
            save_name="bstabdiff_colon",
            save_best=True,  # keep best weights
            use_ema=True,
            ema_decay=ema_decay,
            return_train_info=False,
        )
        return gen

    return _fit


# ============================================================
# 8) Fidelity metric helpers (NaN-safe)
# ============================================================
def _ks_and_wasserstein_1d(x, y):
    """
    Compute KS statistic and 1-Wasserstein distance between 1D samples x,y.
    Pure NumPy; no SciPy dependency.
    """
    x = np.asarray(x, dtype=float)
    y = np.asarray(y, dtype=float)
    nx = len(x)
    ny = len(y)
    if nx == 0 or ny == 0:
        return 0.0, 0.0

    values = np.concatenate([x, y])
    labels = np.concatenate([np.zeros(nx, dtype=int), np.ones(ny, dtype=int)])
    order = np.argsort(values)
    v_sorted = values[order]
    labels_sorted = labels[order]

    c1 = (labels_sorted == 0).astype(float)
    c2 = 1.0 - c1
    cdf1 = np.cumsum(c1) / nx
    cdf2 = np.cumsum(c2) / ny
    diff = cdf1 - cdf2

    ks = float(np.max(np.abs(diff)))

    if len(v_sorted) > 1:
        gaps = np.diff(v_sorted)
        w1 = float(np.sum(np.abs(diff[:-1]) * gaps))
    else:
        w1 = 0.0

    return ks, w1


def _rank_transform(X):
    """
    Simple rank transform per feature (Spearman).
    Assumes mostly continuous data (ties are ignored / broken by order).
    """
    X = np.asarray(X, dtype=float)
    n, m = X.shape
    ranks = np.empty_like(X, dtype=float)
    for j in range(m):
        col = X[:, j]
        order = np.argsort(col)
        ranks[order, j] = np.arange(n, dtype=float)
    return ranks


def _moments_1d(X):
    """
    Compute per-feature mean, var, skewness, kurtosis (standardized 3rd/4th moments).
    """
    X = np.asarray(X, dtype=float)
    mean = X.mean(axis=0)
    centered = X - mean
    var = (centered**2).mean(axis=0)
    eps = 1e-8
    std = np.sqrt(var + eps)
    skew = (centered**3).mean(axis=0) / (std**3)
    kurt = (centered**4).mean(axis=0) / (std**4)
    return mean, var, skew, kurt


def compute_fidelity_metrics(
    X_real,
    y_real,
    X_syn,
    y_syn,
    feature_names=None,
    generator=None,
    random_state=0,
):
    """
    Fidelity diagnostics between real and synthetic tabular data.

    Returns a dict with:
      - ks_per_feature
      - wasserstein_per_feature
      - pearson_mean_abs_corr_diff / max
      - spearman_mean_abs_corr_diff / max
      - mutual info (feature-label) diff stats (if labels provided)
      - higher-order moment diffs
      - optional log-likelihood stats (if generator has log_prob/score_samples)
      - privacy NN distance stats
      - nan_frac_real / nan_frac_syn
    """
    X_real = np.asarray(X_real, dtype=float)
    X_syn = np.asarray(X_syn, dtype=float)

    nan_frac_real = float(np.isnan(X_real).mean())
    nan_frac_syn = float(np.isnan(X_syn).mean())

    # Make sure MI / corr don't crash due to NaNs / infs
    X_real = np.nan_to_num(X_real, nan=0.0, posinf=0.0, neginf=0.0)
    X_syn = np.nan_to_num(X_syn, nan=0.0, posinf=0.0, neginf=0.0)

    assert X_real.shape[1] == X_syn.shape[1], "Real and synthetic must have same #features."
    n_features = X_real.shape[1]

    rng = np.random.RandomState(random_state)

    # 1) KS + Wasserstein per feature
    ks_stats = np.zeros(n_features, dtype=float)
    w1_stats = np.zeros(n_features, dtype=float)
    for j in range(n_features):
        ks_j, w1_j = _ks_and_wasserstein_1d(X_real[:, j], X_syn[:, j])
        ks_stats[j] = ks_j
        w1_stats[j] = w1_j

    # 2) Pairwise correlations (Pearson & Spearman)
    pearson_real = np.corrcoef(X_real, rowvar=False)
    pearson_syn = np.corrcoef(X_syn, rowvar=False)
    mask_offdiag = ~np.eye(n_features, dtype=bool)
    pearson_abs_diff = np.abs(pearson_real - pearson_syn)[mask_offdiag]
    pearson_mean = float(np.mean(pearson_abs_diff))
    pearson_max = float(np.max(pearson_abs_diff))

    ranks_real = _rank_transform(X_real)
    ranks_syn = _rank_transform(X_syn)
    spearman_real = np.corrcoef(ranks_real, rowvar=False)
    spearman_syn = np.corrcoef(ranks_syn, rowvar=False)
    spearman_abs_diff = np.abs(spearman_real - spearman_syn)[mask_offdiag]
    spearman_mean = float(np.mean(spearman_abs_diff))
    spearman_max = float(np.max(spearman_abs_diff))

    # 3) Mutual info (feature-label) diff
    mi_real = mi_syn = mi_abs = None
    mi_mean = mi_max = None
    if y_real is not None and y_syn is not None:
        y_real = np.asarray(y_real).astype(int)
        y_syn = np.asarray(y_syn).astype(int)
        mi_real = mutual_info_classif(X_real, y_real, random_state=random_state)
        mi_syn = mutual_info_classif(X_syn, y_syn, random_state=random_state)
        mi_abs = np.abs(mi_real - mi_syn)
        mi_mean = float(np.mean(mi_abs))
        mi_max = float(np.max(mi_abs))

    # 4) Higher-order moments
    mean_r, var_r, skew_r, kurt_r = _moments_1d(X_real)
    mean_s, var_s, skew_s, kurt_s = _moments_1d(X_syn)
    higher = {
        "mean_abs_diff": float(np.mean(np.abs(mean_r - mean_s))),
        "var_abs_diff": float(np.mean(np.abs(var_r - var_s))),
        "skew_abs_diff": float(np.mean(np.abs(skew_r - skew_s))),
        "kurt_abs_diff": float(np.mean(np.abs(kurt_r - kurt_s))),
    }

    # 5) Likelihood fitness (optional, depends on generator API)
    ll_real = ll_syn = None
    if generator is not None:
        try:
            if hasattr(generator, "log_prob"):
                ll_real = float(np.mean(generator.log_prob(X_real, y_real)))
                ll_syn = float(np.mean(generator.log_prob(X_syn, y_syn)))
            elif hasattr(generator, "score_samples"):
                ll_real = float(np.mean(generator.score_samples(X_real)))
                ll_syn = float(np.mean(generator.score_samples(X_syn)))
        except Exception:
            ll_real = ll_syn = None

    # 6) Privacy leakage via NN distance
    nn_stats = {}
    try:
        nn = NearestNeighbors(n_neighbors=1, metric="euclidean")
        nn.fit(X_real)
        dists, idx = nn.kneighbors(X_syn, return_distance=True)
        dists = dists.ravel()
        nn_stats = {
            "mean_nn_dist": float(np.mean(dists)),
            "min_nn_dist": float(np.min(dists)),
            "max_nn_dist": float(np.max(dists)),
            "p1_nn_dist": float(np.percentile(dists, 1.0)),
            "p5_nn_dist": float(np.percentile(dists, 5.0)),
        }
    except Exception:
        nn_stats = {}

    metrics = {
        "ks_per_feature": ks_stats,
        "wasserstein_per_feature": w1_stats,
        "pearson_mean_abs_corr_diff": pearson_mean,
        "pearson_max_abs_corr_diff": pearson_max,
        "spearman_mean_abs_corr_diff": spearman_mean,
        "spearman_max_abs_corr_diff": spearman_max,
        "higher_order_moment_diffs": higher,
        "privacy_nn_stats": nn_stats,
        "nan_frac_real": nan_frac_real,
        "nan_frac_syn": nan_frac_syn,
    }

    if mi_abs is not None:
        metrics.update(
            {
                "mi_real": mi_real,
                "mi_syn": mi_syn,
                "mi_abs_diff": mi_abs,
                "mi_abs_diff_mean": mi_mean,
                "mi_abs_diff_max": mi_max,
            }
        )
    if ll_real is not None:
        metrics.update(
            {
                "loglik_real_mean": ll_real,
                "loglik_syn_mean": ll_syn,
            }
        )

    return metrics


# ============================================================
# 9) Helper: load Colon dataset
# ============================================================
def load_colon_dataset(
    csv_path: str = "coloncancer_encoded.csv",
    label_col: str = "label",
):
    df = pd.read_csv(csv_path)

    if label_col not in df.columns:
        raise ValueError(
            f"Label column '{label_col}' not found. Columns: {list(df.columns)[:10]} ..."
        )

    y_raw = df[label_col].values
    X_df = df.drop(columns=[label_col])

    # Numeric features only (should already be numeric)
    X_df = X_df.select_dtypes(include=[np.number])
    X = X_df.to_numpy(dtype=np.float32)
    X = np.nan_to_num(X, nan=0.0, posinf=0.0, neginf=0.0)

    # Make labels 0..C-1
    if y_raw.dtype.kind in {"U", "S", "O"}:
        y, _ = pd.factorize(y_raw)
    else:
        unique = np.unique(y_raw)
        if not np.array_equal(unique, np.arange(unique.size)):
            y, _ = pd.factorize(y_raw)
        else:
            y = y_raw.astype(int, copy=False)

    return X, y, list(X_df.columns)


# ============================================================
# 10) Main: strict BSTabDiff + 4 classifiers (no tuning) + fidelity
# ============================================================
if __name__ == "__main__":
    set_seed(GLOBAL_SEED)

    # -------- Load Colon dataset --------
    X, y, feature_names = load_colon_dataset(
        csv_path="coloncancer_encoded.csv",
        label_col="label",
    )
    print(f"Loaded Colon dataset: X.shape={X.shape}, y.shape={y.shape}, classes={np.unique(y)}")

    # -------- Fixed BSTabDiff generator config on cuda:0 --------
    generator_fit_fn = make_generator_fit_fn(
        M=32,
        prior_type="diffusion",
        device="cuda:0",
        prior_epochs=10000,  # careful: used per-fold in strict eval
        prior_batch=128,
        prior_lr=1e-3,
        ema_decay=0.999,
        verbose_every=0,  # set >0 if you want training logs
    )

    # ============================================================
    # (A) Fit BSTabDiff ONCE on full Colon for resource usage + 1 CSV sample
    #     (this global generator is NOT used in strict CV; only for reporting)
    # ============================================================
    proc = psutil.Process(os.getpid()) if psutil is not None else None

    if torch is not None and torch.cuda.is_available():
        torch.cuda.empty_cache()
        torch.cuda.reset_peak_memory_stats()

    cpu_before = proc.memory_info().rss if proc is not None else 0
    t0 = time.time()
    gen_full = generator_fit_fn(X, y, seed=GLOBAL_SEED)
    gen_train_time = time.time() - t0
    cpu_after = proc.memory_info().rss if proc is not None else 0
    cpu_peak = max(cpu_before, cpu_after)
    if torch is not None and torch.cuda.is_available():
        gpu_peak_bytes = torch.cuda.max_memory_allocated()
    else:
        gpu_peak_bytes = 0

    gpu_gib = gpu_peak_bytes / (1024**3) if gpu_peak_bytes > 0 else 0.0
    cpu_gib = cpu_peak / (1024**3) if cpu_peak > 0 else 0.0

    print("\n=== BSTabDiff generator: resource usage (trained ONCE on full Colon) ===")
    print(f"  Train time: {gen_train_time:.2f} sec")
    print(f"  Approx. peak GPU memory: {gpu_gib:.3f} GiB")
    print(f"  Approx. CPU RSS at end: {cpu_gib:.3f} GiB")

    # -------- Sample ONE synthetic Colon dataset from this *global* generator --------
    N_SYN = 200  # you can change this if you want more/less synthetic samples
    X_syn_global, R_syn_global, y_syn_global = gen_full.sample(n=N_SYN)

    save_synth_csv(
        X_syn_global,
        y_syn_global,
        out_csv="colon_synth_BSTabDiff_n200.csv",
        feature_names=feature_names,
    )
    save_synth_mask_csv(R_syn_global, out_csv="colon_synth_BSTabDiff_n200_mask.csv")

    # ============================================================
    # (B) 5x5 CV: strict fold-wise BSTabDiff + 4 classifiers (TSTR/TRTR/C2ST)
    # ============================================================
    classifiers = [
        ("tabpfn", "TabPFN-2.5"),
        ("tandem", "TANDEM"),
        ("catboost", "CatBoost"),
        ("lr", "Logistic Regression"),
    ]

    summary_rows = []

    for clf_name, clf_label in classifiers:
        print(f"\n=== Evaluating STRICT BSTabDiff synthetic + {clf_label} (5x5 TSTR/TRTR/C2ST) ===")

        res = eval_5x5_tstr_foldwise_gen(
            X_real=X,
            y_real=y,
            clf_name=clf_name,
            generator_fit_fn=generator_fit_fn,
            n_syn=N_SYN,
            do_trtr=True,
            do_c2st=True,
            seed=0,
            tabpfn_device="cuda:0" if clf_name == "tabpfn" else None,
            tabpfn_ensembles=16,
            tabpfn_kbest=200 if clf_name == "tabpfn" else None,
        )

        print(f"  TSTR ACC: {res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}")
        print(f"  TSTR AUC: {res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}")
        print(f"  TRTR ACC: {res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}")
        print(f"  TRTR AUC: {res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}")
        print(f"  C2ST ACC: {res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}")
        print(f"  C2ST AUC: {res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}")

        summary_rows.append(
            {
                "Classifier": clf_label,
                "TSTR ACC (mean ± std)": f"{res['tstr_acc_mean']:.4f} ± {res['tstr_acc_std']:.4f}",
                "TSTR AUC (mean ± std)": f"{res['tstr_auc_mean']:.4f} ± {res['tstr_auc_std']:.4f}",
                "TRTR ACC (mean ± std)": f"{res['trtr_acc_mean']:.4f} ± {res['trtr_acc_std']:.4f}",
                "TRTR AUC (mean ± std)": f"{res['trtr_auc_mean']:.4f} ± {res['trtr_auc_std']:.4f}",
                "C2ST ACC (mean ± std)": f"{res['c2st_acc_mean']:.4f} ± {res['c2st_acc_std']:.4f}",
                "C2ST AUC (mean ± std)": f"{res['c2st_auc_mean']:.4f} ± {res['c2st_auc_std']:.4f}",
            }
        )

    summary_df = pd.DataFrame(summary_rows)
    print("\n\n=== Colon: STRICT BSTabDiff synthetic + 4 classifiers: 5x5 TSTR/TRTR/C2ST summary ===")
    print(summary_df.to_markdown(index=False))

    # ============================================================
    # (C) Fidelity diagnostics: real vs global synthetic dataset
    # ============================================================
    fidelity = compute_fidelity_metrics(
        X_real=X,
        y_real=y,
        X_syn=X_syn_global,
        y_syn=y_syn_global,
        feature_names=feature_names,
        generator=gen_full,
        random_state=0,
    )

    print("\n=== Colon: Fidelity metrics (real vs synthetic) ===")
    ks_mean = float(np.mean(fidelity["ks_per_feature"]))
    ks_max = float(np.max(fidelity["ks_per_feature"]))
    w1_mean = float(np.mean(fidelity["wasserstein_per_feature"]))
    w1_max = float(np.max(fidelity["wasserstein_per_feature"]))
    print(f"  KS statistic per feature:    mean={ks_mean:.4f}, max={ks_max:.4f}")
    print(f"  Wasserstein-1 per feature:   mean={w1_mean:.4f}, max={w1_max:.4f}")
    print(
        f"  Pearson |corr_real - corr_syn|:   "
        f"mean={fidelity['pearson_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['pearson_max_abs_corr_diff']:.4f}"
    )
    print(
        f"  Spearman |corr_real - corr_syn|:  "
        f"mean={fidelity['spearman_mean_abs_corr_diff']:.4f}, "
        f"max={fidelity['spearman_max_abs_corr_diff']:.4f}"
    )

    higher = fidelity["higher_order_moment_diffs"]
    print("  Higher-order moments (mean abs diff over features):")
    print(f"    mean:  {higher['mean_abs_diff']:.4e}")
    print(f"    var:   {higher['var_abs_diff']:.4e}")
    print(f"    skew:  {higher['skew_abs_diff']:.4e}")
    print(f"    kurt:  {higher['kurt_abs_diff']:.4e}")

    if "mi_abs_diff_mean" in fidelity:
        print(
            f"  Mutual information (feature-label) |ΔMI|: "
            f"mean={fidelity['mi_abs_diff_mean']:.4f}, "
            f"max={fidelity['mi_abs_diff_max']:.4f}"
        )

    if "loglik_real_mean" in fidelity:
        print(
            f"  Log-likelihood (if available): "
            f"real={fidelity['loglik_real_mean']}, "
            f"synthetic={fidelity['loglik_syn_mean']}"
        )

    print(
        f"  NaN fraction (real): {fidelity['nan_frac_real']:.4e}, "
        f"NaN fraction (synthetic): {fidelity['nan_frac_syn']:.4e}"
    )

    if fidelity["privacy_nn_stats"]:
        nn_s = fidelity["privacy_nn_stats"]
        print("  Privacy (NN distance from synthetic to nearest real):")
        print(
            f"    mean={nn_s['mean_nn_dist']:.4e}, "
            f"min={nn_s['min_nn_dist']:.4e}, "
            f"p1={nn_s['p1_nn_dist']:.4e}, "
            f"p5={nn_s['p5_nn_dist']:.4e}"
        )

Loaded Colon dataset: X.shape=(62, 2000), y.shape=(62,), classes=[0 1]

=== BSTabDiff generator: resource usage (trained ONCE on full Colon) ===
  Train time: 49.20 sec
  Approx. peak GPU memory: 0.025 GiB
  Approx. CPU RSS at end: 2.752 GiB
Saved CSV: colon_synth_BSTabDiff_n200.csv | shape=(200, 2001) | nan_frac=0.00011
Saved mask CSV: colon_synth_BSTabDiff_n200_mask.csv | shape=(200, 2000) | observed_frac=0.99989

=== Evaluating STRICT BSTabDiff synthetic + TabPFN-2.5 (5x5 TSTR/TRTR/C2ST) ===
[Fold 01] TSTR ACC=0.7692, AUC=0.8250
[Fold 02] TSTR ACC=0.7692, AUC=0.8500
[Fold 03] TSTR ACC=0.8333, AUC=0.8750
[Fold 04] TSTR ACC=1.0000, AUC=1.0000
[Fold 05] TSTR ACC=1.0000, AUC=1.0000
[Fold 06] TSTR ACC=0.9231, AUC=0.8750
[Fold 07] TSTR ACC=0.9231, AUC=0.8750
[Fold 08] TSTR ACC=0.6667, AUC=0.7188
[Fold 09] TSTR ACC=0.9167, AUC=0.8750
[Fold 10] TSTR ACC=0.9167, AUC=0.9688
[Fold 11] TSTR ACC=0.8462, AUC=0.8250
[Fold 12] TSTR ACC=0.9231, AUC=0.9750
[Fold 13] TSTR ACC=0.7500, AUC=0.8125
[Fold 

In [ ]:
# ============================================================
# BSTabDiff Ablations (TabPFN-2.5 only)
# ============================================================

import os, time, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.model_selection import RepeatedStratifiedKFold
from sklearn.utils import check_random_state
from sklearn.metrics import roc_auc_score, accuracy_score

# ----------------------------
# 0) Output setup
# ----------------------------
OUT_DIR = "ablations_bstabdiff_tabpfn_compact"
os.makedirs(OUT_DIR, exist_ok=True)

DPI = 300
FIG_EXT = "png"

def _savefig(path):
    plt.tight_layout()
    plt.savefig(path, dpi=DPI, bbox_inches="tight")
    plt.close()

# ----------------------------
# 1) Load dataset (Colon)
# ----------------------------
X, y, feature_names = load_colon_dataset(
    csv_path="coloncancer_encoded.csv",
    label_col="label",
)
print(f"[Data] X={X.shape}, y={y.shape}, classes={np.unique(y)}")

# ----------------------------
# 2) FAST/STRICT mode
# ----------------------------
FAST_MODE = True          # True: 5x2 (10 folds)  | False: 5x5 (25 folds)
N_SPLITS = 5
N_REPEATS = 2 if FAST_MODE else 5

# Synthetic size and TabPFN settings
TABPFN_DEVICE = "cuda:0"
TABPFN_ENSEMBLES = 16
TABPFN_KBEST = 200  # Colon: 2000 feats

# Fidelity snapshot (optional; can be expensive)
RUN_GLOBAL_FIDELITY_SNAPSHOT = False
GLOBAL_FIDELITY_N_SYN = 500

# ----------------------------
# 3) Generator fit fn (ablation knobs)
# ----------------------------
from block_subunit_gen import FeatureSpec, BlockSubunitGenerator, make_equal_blocks

def make_generator_fit_fn_ablation(
    M=32,
    prior_type="diffusion",
    device="cuda:0",
    prior_epochs=3000,
    prior_batch=128,
    prior_lr=1e-3,
    verbose_every=0,
    use_ema=True,
    ema_decay=0.999,
    use_cc_marg=True,
    use_cc_miss=True,
):
    def _fit(Xtr, ytr, seed=0):
        import numpy as np

        # Feature specs (Colon treated as continuous)
        feature_specs = [FeatureSpec(name=f"f{j}", kind="continuous") for j in range(Xtr.shape[1])]

        Xtr = np.asarray(Xtr, dtype=np.float32)
        ytr = np.asarray(ytr).astype(int)
        R = np.isfinite(Xtr).astype(np.int64)

        blocks = make_equal_blocks(m=Xtr.shape[1], M=M)
        n_classes = int(ytr.max()) + 1

        gen = BlockSubunitGenerator(
            feature_specs=feature_specs,
            blocks=blocks,
            n_classes=n_classes,
            prior_type=prior_type,
            device=device,
            use_class_cond_marginals=bool(use_cc_marg),
            use_class_cond_missingness=bool(use_cc_miss),
        )
        gen.set_permutation(None)

        gen.fit_marginals(X=Xtr, R=R, y=ytr)
        h_hat = gen.infer_h(X=Xtr, R=R, y=ytr)
        gen.fit_emissions(X=Xtr, R=R, y=ytr, h_hat=h_hat)

        _ = gen.train_prior(
            h_hat=h_hat,
            y=ytr,
            epochs=int(prior_epochs),
            batch_size=int(prior_batch),
            lr=float(prior_lr),
            verbose_every=int(verbose_every),
            save_dir=None,
            save_name="bstabdiff_ablation",
            save_best=True,
            use_ema=bool(use_ema),
            ema_decay=float(ema_decay),
            return_train_info=False,
        )
        return gen
    return _fit

# ----------------------------
# 4) Minimal CV evaluator (TabPFN only) with configurable repeats
# ----------------------------
def eval_cv_tstr_foldwise_gen_tabpfn(
    X_real,
    y_real,
    generator_fit_fn,
    n_syn=200,
    do_trtr=True,
    do_c2st=True,
    seed=0,
    n_splits=5,
    n_repeats=5,
    tabpfn_device="cuda:0",
    tabpfn_ensembles=16,
    tabpfn_kbest=200,
):
    # uses your make_clf + _predict_proba_binary from earlier code
    # (assumes make_clf("tabpfn", ...) exists in notebook)
    X_real = np.asarray(X_real)
    y_real = np.asarray(y_real).astype(int)

    rng = check_random_state(seed)
    rkf = RepeatedStratifiedKFold(n_splits=n_splits, n_repeats=n_repeats, random_state=seed)

    tstr_auc, tstr_acc = [], []
    trtr_auc, trtr_acc = [], []
    c2st_auc, c2st_acc = [], []

    for fold_id, (tr, te) in enumerate(rkf.split(X_real, y_real), 1):
        Xtr, ytr = X_real[tr], y_real[tr]
        Xte, yte = X_real[te], y_real[te]

        fold_seed = int(rng.randint(0, 2**31 - 1))

        # (1) Fit generator on train only
        gen_fold = generator_fit_fn(Xtr, ytr, seed=fold_seed)
        X_syn, R_syn, y_syn = gen_fold.sample(n=n_syn)

        # (2) TSTR
        clf_tstr = make_clf(
            "tabpfn",
            seed=fold_seed,
            tabpfn_device=tabpfn_device,
            tabpfn_ensembles=tabpfn_ensembles,
            tabpfn_kbest=tabpfn_kbest,
        )
        clf_tstr.fit(X_syn, y_syn)
        proba_t = _predict_proba_binary(clf_tstr, Xte)
        pred_t = (proba_t >= 0.5).astype(int)
        tstr_auc.append(roc_auc_score(yte, proba_t))
        tstr_acc.append(accuracy_score(yte, pred_t))

        # (3) TRTR
        if do_trtr:
            clf_trtr = make_clf(
                "tabpfn",
                seed=fold_seed,
                tabpfn_device=tabpfn_device,
                tabpfn_ensembles=tabpfn_ensembles,
                tabpfn_kbest=tabpfn_kbest,
            )
            clf_trtr.fit(Xtr, ytr)
            proba_r = _predict_proba_binary(clf_trtr, Xte)
            pred_r = (proba_r >= 0.5).astype(int)
            trtr_auc.append(roc_auc_score(yte, proba_r))
            trtr_acc.append(accuracy_score(yte, pred_r))

        # (4) C2ST (lightweight: LR discriminator)
        if do_c2st:
            n_mix = min(len(Xtr), len(X_syn))
            idx_r = rng.choice(len(Xtr), size=n_mix, replace=False)
            idx_s = rng.choice(len(X_syn), size=n_mix, replace=False)

            Xmix = np.vstack([Xtr[idx_r], X_syn[idx_s]])
            ymix = np.concatenate([np.ones(n_mix, dtype=int), np.zeros(n_mix, dtype=int)])
            p = rng.permutation(len(ymix))
            Xmix, ymix = Xmix[p], ymix[p]

            cut = int(0.8 * len(ymix))
            Xmix_tr, Xmix_te = Xmix[:cut], Xmix[cut:]
            ymix_tr, ymix_te = ymix[:cut], ymix[cut:]

            clf_c2st = make_clf("lr", seed=fold_seed)
            clf_c2st.fit(Xmix_tr, ymix_tr)
            proba_m = _predict_proba_binary(clf_c2st, Xmix_te)
            pred_m = (proba_m >= 0.5).astype(int)
            c2st_auc.append(roc_auc_score(ymix_te, proba_m))
            c2st_acc.append(accuracy_score(ymix_te, pred_m))

    out = {
        "n_folds": int(len(tstr_auc)),
        "tstr_auc_mean": float(np.mean(tstr_auc)),
        "tstr_auc_std": float(np.std(tstr_auc)),
        "tstr_acc_mean": float(np.mean(tstr_acc)),
        "tstr_acc_std": float(np.std(tstr_acc)),
    }
    if do_trtr and len(trtr_auc) > 0:
        out.update(
            {
                "trtr_auc_mean": float(np.mean(trtr_auc)),
                "trtr_auc_std": float(np.std(trtr_auc)),
                "trtr_acc_mean": float(np.mean(trtr_acc)),
                "trtr_acc_std": float(np.std(trtr_acc)),
            }
        )
    if do_c2st and len(c2st_auc) > 0:
        out.update(
            {
                "c2st_auc_mean": float(np.mean(c2st_auc)),
                "c2st_auc_std": float(np.std(c2st_auc)),
                "c2st_acc_mean": float(np.mean(c2st_acc)),
                "c2st_acc_std": float(np.std(c2st_acc)),
            }
        )
    return out

# ----------------------------
# 5) Compact ablation set (~11 configs)
#    Baseline + one-factor-at-a-time changes (ablation story)
# ----------------------------
BASE = dict(
    prior_type="diffusion",
    M=32,
    prior_epochs=3000,
    n_syn=200,
    use_ema=True,
    use_cc_marg=True,
    use_cc_miss=True,
    device="cuda:0",
    prior_batch=128,
    prior_lr=1e-3,
    ema_decay=0.999,
)

ABLATIONS = [
    dict(name="BASE", **BASE),

    # Core component ablations
    dict(name="noEMA", **{**BASE, "use_ema": False}),
    dict(name="noCC-Marg", **{**BASE, "use_cc_marg": False}),
    dict(name="noCC-Miss", **{**BASE, "use_cc_miss": False}),

    # Prior ablation
    dict(name="FlowPrior", **{**BASE, "prior_type": "flow", "use_ema": False}),  # EMA irrelevant

    # Scaling knobs
    dict(name="M16", **{**BASE, "M": 16}),
    dict(name="M64", **{**BASE, "M": 64}),
    dict(name="nSyn100", **{**BASE, "n_syn": 100}),
    dict(name="nSyn500", **{**BASE, "n_syn": 500}),
    dict(name="ep1500", **{**BASE, "prior_epochs": 1500}),
    dict(name="ep5000", **{**BASE, "prior_epochs": 5000}),
]

print(f"[Ablations] n={len(ABLATIONS)} (FAST_MODE={FAST_MODE}, CV={N_SPLITS}x{N_REPEATS} => folds={N_SPLITS*N_REPEATS})")
print(json.dumps(ABLATIONS[:3], indent=2))

# ----------------------------
# 6) Run ablations
# ----------------------------
rows = []
t_start = time.time()

for i, cfg in enumerate(ABLATIONS, 1):
    tag = f"{i:02d}_{cfg['name']}_prior={cfg['prior_type']}_M={cfg['M']}_ep={cfg['prior_epochs']}_nsyn={cfg['n_syn']}_ema={int(cfg['use_ema'])}_ccm={int(cfg['use_cc_marg'])}_ccmiss={int(cfg['use_cc_miss'])}"
    print(f"\n[{i}/{len(ABLATIONS)}] {tag}")

    generator_fit_fn = make_generator_fit_fn_ablation(
        M=cfg["M"],
        prior_type=cfg["prior_type"],
        device=cfg["device"],
        prior_epochs=cfg["prior_epochs"],
        prior_batch=cfg["prior_batch"],
        prior_lr=cfg["prior_lr"],
        verbose_every=0,
        use_ema=cfg["use_ema"],
        ema_decay=cfg["ema_decay"],
        use_cc_marg=cfg["use_cc_marg"],
        use_cc_miss=cfg["use_cc_miss"],
    )

    t0 = time.time()
    res = eval_cv_tstr_foldwise_gen_tabpfn(
        X_real=X,
        y_real=y,
        generator_fit_fn=generator_fit_fn,
        n_syn=cfg["n_syn"],
        do_trtr=True,
        do_c2st=True,
        seed=0,
        n_splits=N_SPLITS,
        n_repeats=N_REPEATS,
        tabpfn_device=TABPFN_DEVICE,
        tabpfn_ensembles=TABPFN_ENSEMBLES,
        tabpfn_kbest=TABPFN_KBEST,
    )
    dt = time.time() - t0

    eff_auc_ratio = res["tstr_auc_mean"] / max(res["trtr_auc_mean"], 1e-12)
    eff_auc_delta = res["tstr_auc_mean"] - res["trtr_auc_mean"]

    fid = {}
    if RUN_GLOBAL_FIDELITY_SNAPSHOT:
        try:
            gen_full = generator_fit_fn(X, y, seed=123)
            X_syn_g, R_syn_g, y_syn_g = gen_full.sample(n=GLOBAL_FIDELITY_N_SYN)
            fidelity = compute_fidelity_metrics(
                X_real=X, y_real=y,
                X_syn=X_syn_g, y_syn=y_syn_g,
                feature_names=feature_names,
                generator=gen_full,
                random_state=0,
            )
            fid = {
                "ks_mean": float(np.mean(fidelity["ks_per_feature"])),
                "c2st_auc_mean_global_dummy": np.nan,  # keep schema stable if you want
            }
        except Exception as e:
            print(f"  [Warn] Fidelity snapshot failed: {repr(e)}")

    rows.append({
        "tag": tag,
        "name": cfg["name"],
        **cfg,
        "runtime_sec": float(dt),
        "folds": int(res["n_folds"]),
        "tstr_auc_mean": res["tstr_auc_mean"],
        "tstr_auc_std": res["tstr_auc_std"],
        "tstr_acc_mean": res["tstr_acc_mean"],
        "tstr_acc_std": res["tstr_acc_std"],
        "trtr_auc_mean": res["trtr_auc_mean"],
        "trtr_auc_std": res["trtr_auc_std"],
        "trtr_acc_mean": res["trtr_acc_mean"],
        "trtr_acc_std": res["trtr_acc_std"],
        "c2st_auc_mean": res["c2st_auc_mean"],
        "c2st_auc_std": res["c2st_auc_std"],
        "c2st_acc_mean": res["c2st_acc_mean"],
        "c2st_acc_std": res["c2st_acc_std"],
        "eff_auc_ratio": float(eff_auc_ratio),
        "eff_auc_delta": float(eff_auc_delta),
        **fid,
    })

df = pd.DataFrame(rows).sort_values("tstr_auc_mean", ascending=False).reset_index(drop=True)
csv_path = os.path.join(OUT_DIR, "ablations_compact_results.csv")
df.to_csv(csv_path, index=False)

print(f"\n[Saved] {csv_path}")
print(f"[Done] total wall time: {time.time() - t_start:.1f}s")

# ----------------------------
# 7) Plots (all 300 dpi)
# ----------------------------
# (A) AUC bar (all configs, small n => readable)
plt.figure(figsize=(12, 4.8))
x = np.arange(len(df))
plt.bar(x, df["tstr_auc_mean"].values, yerr=df["tstr_auc_std"].values)
plt.xticks(x, df["name"].values, rotation=45, ha="right")
plt.ylabel("TSTR AUC (TabPFN-2.5)")
plt.title(f"BSTabDiff Ablations: TSTR AUC (mean ± std over {int(df.loc[0,'folds'])} folds)")
_savefig(os.path.join(OUT_DIR, f"ablation_tstr_auc_all.{FIG_EXT}"))

# (B) Efficiency ratio bar
plt.figure(figsize=(12, 4.8))
plt.bar(x, df["eff_auc_ratio"].values)
plt.axhline(1.0, linestyle="--")
plt.xticks(x, df["name"].values, rotation=45, ha="right")
plt.ylabel("AUC Efficiency = TSTR / TRTR")
plt.title("BSTabDiff Ablations: AUC Efficiency (TabPFN-2.5)")
_savefig(os.path.join(OUT_DIR, f"ablation_auc_eff_ratio_all.{FIG_EXT}"))

# (C) Tradeoff: TSTR AUC vs C2ST AUC
plt.figure(figsize=(6.5, 5.5))
plt.scatter(df["c2st_auc_mean"].values, df["tstr_auc_mean"].values)
for _, r in df.iterrows():
    plt.annotate(r["name"], (r["c2st_auc_mean"], r["tstr_auc_mean"]), fontsize=8, xytext=(4, 3), textcoords="offset points")
plt.axvline(0.5, linestyle="--")
plt.xlabel("C2ST AUC (real vs synth; 0.5 ideal)")
plt.ylabel("TSTR AUC (TabPFN-2.5)")
plt.title("Utility vs Detectability (Compact Ablations)")
_savefig(os.path.join(OUT_DIR, f"ablation_tradeoff_tstr_vs_c2st.{FIG_EXT}"))

# (D) “One-factor” curves (only if those points exist)
def _plot_curve(names_in_order, xvals, xlabel, title, outname):
    dd = df.set_index("name")
    keep = [n for n in names_in_order if n in dd.index]
    if len(keep) < 2:
        return
    ymean = [dd.loc[n, "tstr_auc_mean"] for n in keep]
    ystd  = [dd.loc[n, "tstr_auc_std"] for n in keep]
    plt.figure(figsize=(6.3, 4.6))
    plt.plot(xvals[:len(keep)], ymean, marker="o")
    ymean = np.array(ymean); ystd = np.array(ystd)
    plt.fill_between(xvals[:len(keep)], ymean-ystd, ymean+ystd, alpha=0.2)
    plt.xticks(xvals[:len(keep)], xvals[:len(keep)])
    plt.xlabel(xlabel)
    plt.ylabel("TSTR AUC")
    plt.title(title)
    _savefig(os.path.join(OUT_DIR, f"{outname}.{FIG_EXT}"))

# M curve: M16 -> BASE -> M64
_plot_curve(["M16", "BASE", "M64"], [16, 32, 64], "#Blocks (M)", "Scaling: TSTR AUC vs M", "curve_tstr_vs_M")

# n_syn curve: nSyn100 -> BASE -> nSyn500
_plot_curve(["nSyn100", "BASE", "nSyn500"], [100, 200, 500], "Synthetic samples per fold", "Scaling: TSTR AUC vs n_syn", "curve_tstr_vs_nsyn")

# epochs curve: ep1500 -> BASE -> ep5000
_plot_curve(["ep1500", "BASE", "ep5000"], [1500, 3000, 5000], "Prior training epochs", "Scaling: TSTR AUC vs prior_epochs", "curve_tstr_vs_epochs")

print("\n[Saved figures to]", OUT_DIR)
print("Outputs:")
print(" - ablations_compact_results.csv")
print(f" - *.png (all at {DPI} dpi)")

[Data] X=(62, 2000), y=(62,), classes=[0 1]
[Ablations] n=11 (FAST_MODE=True, CV=5x2 => folds=10)
[
  {
    "name": "BASE",
    "prior_type": "diffusion",
    "M": 32,
    "prior_epochs": 3000,
    "n_syn": 200,
    "use_ema": true,
    "use_cc_marg": true,
    "use_cc_miss": true,
    "device": "cuda:0",
    "prior_batch": 128,
    "prior_lr": 0.001,
    "ema_decay": 0.999
  },
  {
    "name": "noEMA",
    "prior_type": "diffusion",
    "M": 32,
    "prior_epochs": 3000,
    "n_syn": 200,
    "use_ema": false,
    "use_cc_marg": true,
    "use_cc_miss": true,
    "device": "cuda:0",
    "prior_batch": 128,
    "prior_lr": 0.001,
    "ema_decay": 0.999
  },
  {
    "name": "noCC-Marg",
    "prior_type": "diffusion",
    "M": 32,
    "prior_epochs": 3000,
    "n_syn": 200,
    "use_ema": true,
    "use_cc_marg": false,
    "use_cc_miss": true,
    "device": "cuda:0",
    "prior_batch": 128,
    "prior_lr": 0.001,
    "ema_decay": 0.999
  }
]

[1/11] 01_BASE_prior=diffusion_M=32_ep=3000_